### Topic: Practical Example of RunnableLambda

In [ ]:
from langchain_core.runnables import RunnableLambda


# ---------------------------------------------------------
# Step 1: Define a normal Python function
# ---------------------------------------------------------

def clean_text(text):
    """
    Remove unnecessary spaces
    and convert text to lowercase.
    """

    return text.strip().lower()


# ---------------------------------------------------------
# Step 2: Convert the function into a Runnable
# ---------------------------------------------------------

clean_text_runnable = RunnableLambda(clean_text)


# ---------------------------------------------------------
# Step 3: Execute the Runnable
# ---------------------------------------------------------

result = clean_text_runnable.invoke(
    "   Generative AI   "
)


# ---------------------------------------------------------
# Step 4: Display the result
# ---------------------------------------------------------

print(result)

In [ ]:
from langchain_core.runnables import RunnableLambda


# ---------------------------------------------------------
# Step 1: Define the first function
# ---------------------------------------------------------

def clean_text(text):
    """Remove leading and trailing spaces."""
    return text.strip()


# ---------------------------------------------------------
# Step 2: Define the second function
# ---------------------------------------------------------

def add_instruction(text):
    """Add an instruction before the topic."""
    return f"Explain this topic in simple language: {text}"


# ---------------------------------------------------------
# Step 3: Convert functions into Runnables
# ---------------------------------------------------------

clean_runnable = RunnableLambda(clean_text)

instruction_runnable = RunnableLambda(add_instruction)


# ---------------------------------------------------------
# Step 4: Compose them sequentially
# ---------------------------------------------------------

chain = clean_runnable | instruction_runnable


# ---------------------------------------------------------
# Step 5: Execute the chain
# ---------------------------------------------------------

result = chain.invoke(
    "   Generative AI   "
)


print(result)

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# Problem: Retriever returns list[Document], but prompt expects a string
# Solution: RunnableLambda to reshape

def format_docs(docs):
    """Convert list of Documents into a single context string."""
    return "\n\n---\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),  # ← Reshape!
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)


# After a parallel chain, you get a dict. Extract what you need.
parallel_chain = {
    "summary": summary_chain,
    "sentiment": sentiment_chain
}

# Extract just the summary for the next step
chain = (
    parallel_chain
    | RunnableLambda(lambda x: x["summary"])  # ← Extract field
    | RunnableLambda(lambda text: text[:200])  # ← Truncate
)



def merge_analysis(data: dict) -> str:
    """Merge parallel analysis results into a single report."""
    return (
        f"📋 SUMMARY: {data['summary']}\n\n"
        f"💬 SENTIMENT: {data['sentiment']}\n\n"
        f"🏷️ KEYWORDS: {data['keywords']}\n\n"
        f"⚠️ RISKS: {data['risks']}"
    )

chain = (
    RunnableParallel(
        summary=summary_chain,
        sentiment=sentiment_chain,
        keywords=keywords_chain,
        risks=risk_chain
    )
    | RunnableLambda(merge_analysis)  # ← Merge all outputs
)

In [ ]:
### Real-World Example 1 — Data Transformation Pipeline
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# ─── Transformation Functions ───

def remove_markdown(text: str) -> str:
    """Strip markdown formatting from LLM output."""
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)  # Bold
    text = re.sub(r'\*(.*?)\*', r'\1', text)       # Italic
    text = re.sub(r'`(.*?)`', r'\1', text)         # Code
    text = re.sub(r'#{1,6}\s', '', text)            # Headers
    return text.strip()

def extract_bullet_points(text: str) -> list[str]:
    """Extract bullet points from text."""
    bullets = re.findall(r'[-•]\s*(.+)', text)
    return [b.strip() for b in bullets]

def count_statistics(text: str) -> dict:
    """Compute text statistics."""
    words = text.split()
    sentences = re.split(r'[.!?]+', text)
    return {
        "word_count": len(words),
        "sentence_count": len([s for s in sentences if s.strip()]),
        "avg_words_per_sentence": round(len(words) / max(len(sentences), 1), 1),
        "char_count": len(text)
    }

def format_report(data: dict) -> str:
    """Format the final enriched output."""
    return (
        f"📝 CLEAN TEXT:\n{data['clean']}\n\n"
        f"📊 STATS: {data['stats']['word_count']} words, "
        f"{data['stats']['sentence_count']} sentences\n\n"
        f"🔹 BULLET POINTS ({len(data['bullets'])}):\n"
        + "\n".join(f"  • {b}" for b in data['bullets'])
    )

# ─── Build the Pipeline ───
chain = (
    ChatPromptTemplate.from_template("List 5 benefits of {topic}. Use bullet points.")
    | llm
    | StrOutputParser()
    | RunnableLambda(remove_markdown)            # Step 1: Clean
    | RunnableLambda(lambda text: {              # Step 2: Enrich
        "clean": text,
        "bullets": extract_bullet_points(text),
        "stats": count_statistics(text)
    })
    | RunnableLambda(format_report)              # Step 3: Format
)

result = chain.invoke({"topic": "remote work"})
print(result)
# 📝 CLEAN TEXT:
# Here are five benefits of remote work...
#
# 📊 STATS: 85 words, 7 sentences
#
# 🔹 BULLET POINTS (5):
#   • Increased flexibility and work-life balance
#   • Reduced commuting time and costs
#   • Access to a global talent pool
#   • Higher productivity and focus
#   • Lower overhead costs for employers

In [ ]:
# Real-World Example 3 — External API Enrichment
import requests
from langchain_core.runnables import RunnableLambda

def enrich_with_weather(text: str) -> str:
    """Add current weather data to the LLM response."""
    try:
        # Call external weather API
        response = requests.get(
            "https://wttr.in/?format=%C+%t",
            timeout=3
        )
        weather = response.text.strip()
        return f"{text}\n\n🌤️ Current weather: {weather}"
    except Exception:
        return f"{text}\n\n🌤️ Weather: unavailable"

def add_timestamp(text: str) -> str:
    """Add a timestamp to the response."""
    from datetime import datetime
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"[{now}]\n{text}"

chain = (
    ChatPromptTemplate.from_template("Give me a travel tip for {city}")
    | llm
    | StrOutputParser()
    | RunnableLambda(enrich_with_weather)  # ← External API call!
    | RunnableLambda(add_timestamp)        # ← Add metadata!
)

result = chain.invoke({"city": "Paris"})
print(result)
# [2025-01-15 14:32:01]
# When visiting Paris, make sure to explore the Marais district...
#
# 🌤️ Current weather: Partly cloudy +8°C